# Semana 3 — Puertas Cuánticas de Un Qubit
## Las operaciones que transforman el estado de un qubit

**Objetivo:** Dominar las puertas de un qubit (X, Y, Z, H, S, T, Rx, Ry, Rz) y entender que son rotaciones en la esfera de Bloch.

**Hito verificable:** Puedes aplicar cualquier secuencia de puertas a un qubit, verificar que el resultado es unitario y visualizar la trayectoria en la esfera de Bloch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
print('Librerías importadas correctamente')

## Ejercicio 3.1 — Las puertas de Pauli: X, Y, Z
Son las puertas más fundamentales. X es el NOT cuántico; Y y Z añaden fases.

In [ ]:
# Las matrices de Pauli
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)  # NOT cuántico
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

print('=== Puerta X (NOT cuántico) ===')
print(f'X|0⟩ = {X @ ket_0}  → debe ser |1⟩ = [0, 1]')
print(f'X|1⟩ = {X @ ket_1}  → debe ser |0⟩ = [1, 0]')
print()
print('=== Puerta Z (cambio de fase) ===')
print(f'Z|0⟩ = {Z @ ket_0}  → |0⟩ sin cambio')
print(f'Z|1⟩ = {Z @ ket_1}  → -|1⟩ (fase π)')
print()
print('=== Puerta Y ===')
print(f'Y|0⟩ = {Y @ ket_0}  → i|1⟩')
print(f'Y|1⟩ = {Y @ ket_1}  → -i|0⟩')
print()

# Verificar unitariedad
for nombre, G in [('X', X), ('Y', Y), ('Z', Z)]:
    es_unit = np.allclose(G.conj().T @ G, I)
    print(f'{nombre}†{nombre} = I → {es_unit}')

## Ejercicio 3.2 — La puerta Hadamard y la base de Fourier
La puerta H crea superposición. Es su propio inverso: HH = I.

In [ ]:
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

print('=== Puerta Hadamard ===')
print(f'H|0⟩ = {H @ ket_0}  → |+⟩')
print(f'H|1⟩ = {H @ ket_1}  → |−⟩')
print(f'H|+⟩ = {H @ (H @ ket_0)}  → |0⟩ (H es su propio inverso)')
print(f'H|−⟩ = {H @ (H @ ket_1)}  → |1⟩')
print()
print(f'HH = I → {np.allclose(H @ H, I)}')
print()
print('INTUICIÓN: H rota 180° alrededor del eje diagonal X+Z en la esfera de Bloch.')
print('Convierte la base computacional {|0⟩,|1⟩} en la base de Hadamard {|+⟩,|−⟩}.')

## Ejercicio 3.3 — Puertas de fase: S y T
S = Z^(1/2) y T = Z^(1/4). Añaden una fase relativa a |1⟩ sin cambiar las probabilidades.

In [ ]:
S = np.array([[1, 0], [0, 1j]], dtype=complex)         # fase π/2
T = np.array([[1, 0], [0, np.exp(1j*np.pi/4)]], dtype=complex)  # fase π/4

ket_plus = (ket_0 + ket_1) / np.sqrt(2)

estados = {
    '|0⟩': ket_0,
    '|1⟩': ket_1,
    '|+⟩': ket_plus,
}

for nombre_g, G in [('S', S), ('T', T)]:
    print(f'=== Puerta {nombre_g} ===')
    for nombre_e, psi in estados.items():
        resultado = G @ psi
        probs_antes = np.abs(psi)**2
        probs_despues = np.abs(resultado)**2
        print(f'  {nombre_g}{nombre_e}: {np.round(resultado, 4)}')
        print(f'    Probs antes:   {np.round(probs_antes, 4)}')
        print(f'    Probs después: {np.round(probs_despues, 4)}  (sin cambio)')
    print()

print('CLAVE: Las puertas de fase no cambian las probabilidades de medición en la base Z.')
print('Pero sí cambian el estado — la fase importará cuando haya interferencia (Semana 4).')

## Ejercicio 3.4 — Puertas de rotación Rx, Ry, Rz
Toda puerta de un qubit es una rotación en la esfera de Bloch.

In [ ]:
def Rx(theta):
    """Rotación θ alrededor del eje X."""
    c, s = np.cos(theta/2), np.sin(theta/2)
    return np.array([[c, -1j*s], [-1j*s, c]], dtype=complex)

def Ry(theta):
    """Rotación θ alrededor del eje Y."""
    c, s = np.cos(theta/2), np.sin(theta/2)
    return np.array([[c, -s], [s, c]], dtype=complex)

def Rz(theta):
    """Rotación θ alrededor del eje Z."""
    return np.array([[np.exp(-1j*theta/2), 0], [0, np.exp(1j*theta/2)]], dtype=complex)

# Verificar que las puertas conocidas son casos especiales de Rx/Ry/Rz
print('=== Verificar relaciones ===')
print(f'Rx(π) ≈ iX: {np.allclose(Rx(np.pi), 1j*X)}')
print(f'Ry(π) ≈ iY: {np.allclose(Ry(np.pi), 1j*Y)}')
print(f'Rz(π) ≈ iZ: {np.allclose(Rz(np.pi), 1j*Z)}')
print(f'Ry(π/2) convierte |0⟩ en |+⟩: {np.allclose(Ry(np.pi/2) @ ket_0, (ket_0+ket_1)/np.sqrt(2))}')
print()

# Demostración: cualquier puerta es Rz·Ry·Rz (descomposición ZYZ)
alpha, beta, gamma = np.pi/3, np.pi/4, np.pi/6
U_compuesto = Rz(alpha) @ Ry(beta) @ Rz(gamma)
print(f'Puerta arbitraria U = Rz(π/3)·Ry(π/4)·Rz(π/6):')
print(np.round(U_compuesto, 4))
print(f'Es unitaria: {np.allclose(U_compuesto.conj().T @ U_compuesto, I)}')

## Ejercicio 3.5 — La esfera de Bloch
Todo estado de un qubit se puede representar como un punto en la esfera de Bloch.

In [ ]:
def estado_a_bloch(psi):
    """Convierte un estado cuántico en coordenadas de la esfera de Bloch (x, y, z)."""
    # Parametrización: |ψ⟩ = cos(θ/2)|0⟩ + e^(iφ)sin(θ/2)|1⟩
    # x = sin(θ)cos(φ), y = sin(θ)sin(φ), z = cos(θ)
    # Equivalente: x=⟨X⟩, y=⟨Y⟩, z=⟨Z⟩
    x = 2 * np.real(psi[0].conj() * psi[1])
    y = 2 * np.imag(psi[0].conj() * psi[1])
    z = abs(psi[0])**2 - abs(psi[1])**2
    return x, y, z

# Estados conocidos y sus coordenadas Bloch
estados_bloch = [
    ('|0⟩',  np.array([1, 0], dtype=complex)),
    ('|1⟩',  np.array([0, 1], dtype=complex)),
    ('|+⟩',  np.array([1, 1], dtype=complex) / np.sqrt(2)),
    ('|−⟩',  np.array([1, -1], dtype=complex) / np.sqrt(2)),
    ('|+i⟩', np.array([1, 1j], dtype=complex) / np.sqrt(2)),
    ('|−i⟩', np.array([1, -1j], dtype=complex) / np.sqrt(2)),
]

print(f'{"Estado":>6}  {"x":>8}  {"y":>8}  {"z":>8}')
print('-' * 40)
for nombre, psi in estados_bloch:
    x, y, z = estado_a_bloch(psi)
    print(f'{nombre:>6}  {x:>8.4f}  {y:>8.4f}  {z:>8.4f}')

print()
print('REGLA: El polo norte es |0⟩ (z=1), el sur es |1⟩ (z=-1).')
print('Los estados del ecuador son superposición 50/50 con diferentes fases.')

## Mini reto — Completar antes de avanzar a la Semana 4

Implementa `aplicar_circuito(psi, puertas)` que reciba un estado inicial y una lista de puertas (matrices 2×2), las aplique en orden, y muestre:
- El estado tras cada puerta
- Las coordenadas de Bloch en cada paso
- Que el estado final sigue siendo válido (norma = 1)

Pruébalo con la secuencia H → S → H → T que convierte |0⟩ en un estado con fase compleja.

In [ ]:
# TU CÓDIGO AQUÍ
def aplicar_circuito(psi: np.ndarray, puertas: list) -> np.ndarray:
    """Aplica una lista de puertas cuánticas al estado psi."""
    # TODO: implementar
    pass

## ✅ Hito de la Semana 3

Antes de avanzar, comprueba que puedes:

- [ ] Construir y aplicar las 6 puertas fundamentales: X, Y, Z, H, S, T
- [ ] Verificar que todas son unitarias
- [ ] Calcular coordenadas de Bloch de cualquier estado
- [ ] Implementar Rx(θ), Ry(θ), Rz(θ) y verificar sus casos límite
- [ ] Componer múltiples puertas y obtener el estado final

## Reflexión (escribe aquí tu respuesta)

**¿Por qué todas las puertas cuánticas deben ser matrices unitarias?**

*(tu respuesta aquí)*

**¿Cuál es la diferencia entre la puerta X y la puerta NOT clásica?**

*(tu respuesta aquí)*

---
*Siguiente: Semana 4 — Interferencia Cuántica y Fases*